# 06 - Model Evaluation (Classification + Segmentation)

Đánh giá mô hình classification và segmentation trên test set, xuất báo cáo và metrics cuối cùng.

In [12]:
import os
import sys
proj_root = r'E:\Master\thesis_durian'
sys.path.insert(0, proj_root)
sys.path.insert(0, os.path.join(proj_root, 'utils'))

import torch
import json
import torch.nn as nn

from utils.preprocessing import DataLoader
from utils.models import EfficientNetClassifier, EfficientNetUNet
from utils.metrics import ModelEvaluator, calculate_segmentation_metrics
from torch.utils.data import DataLoader as TorchDataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

device cpu


## 1) Load test data

In [4]:
data_loader = DataLoader('data/raw/test', batch_size=8, image_size=(224,224))
splits = data_loader.load_dataset()
test_loader = data_loader.get_data_loader('test', is_training=False)
print('Test size', len(splits['test']))

Test size 89


## 2) Evaluate classification model

In [ ]:
num_classes = len(data_loader.get_class_distribution())
clf = EfficientNetClassifier(num_classes=num_classes, backbone='efficientnet_b0', pretrained=False).to(device)
import glob
checkpoint = max(glob.glob('models/classification/checkpoints/efficientnet_baseline_best_*.pth'), key=os.path.getctime)
state = torch.load(checkpoint, map_location=device)
clf.load_state_dict(state['model_state_dict'])
clf.eval()
evaluator = ModelEvaluator(device=device)
clf_metrics = evaluator.evaluate_classification(clf, test_loader)
print('Classification test metrics:', clf_metrics)
os.makedirs('models/classification/metrics', exist_ok=True)
with open('models/classification/metrics/final_classification_metrics.json', 'w') as f:
    json.dump(clf_metrics, f, indent=2)

Classification test metrics: {'accuracy': 0.8876404494382022, 'precision': 0.8969278229722798, 'recall': 0.8876404494382022, 'f1_score': 0.8874887252178796, 'precision_per_class': [0.7619047619047619, 0.9473684210526315, 0.9411764705882353, 0.9444444444444444, 0.8571428571428571], 'recall_per_class': [0.9411764705882353, 0.8571428571428571, 1.0, 0.7727272727272727, 0.9230769230769231], 'f1_per_class': [0.8421052631578947, 0.9, 0.9696969696969697, 0.85, 0.8888888888888888], 'confusion_matrix': [[16, 1, 0, 0, 0], [1, 18, 1, 1, 0], [0, 0, 16, 0, 0], [3, 0, 0, 17, 2], [1, 0, 0, 0, 12]], 'num_samples': 89, 'num_classes': 5}


## 3) Evaluate segmentation model

In [10]:
# ============================================================
# 2. METRICS
# ============================================================
def calculate_segmentation_metrics(pred, gt, smooth=1e-6):
    """pred, gt: numpy array binary (0/1) shape [H, W]"""
    pred = pred.flatten().astype(bool)
    gt   = gt.flatten().astype(bool)

    tp = (pred & gt).sum()
    fp = (pred & ~gt).sum()
    fn = (~pred & gt).sum()

    precision = (tp + smooth) / (tp + fp + smooth)
    recall    = (tp + smooth) / (tp + fn + smooth)
    f1        = 2 * precision * recall / (precision + recall + smooth)
    dice      = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)

    return {'iou': float(iou), 'dice': float(dice),
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1)}


def evaluate_metrics(model, loader, criterion_bce, dice_loss_fn, device):
    """Chạy full validation, trả về loss + tất cả metrics"""
    model.eval()
    total_loss = 0.0
    all_metrics = {'iou': [], 'dice': [], 'precision': [], 'recall': [], 'f1': []}

    with torch.no_grad():
        for batch in loader:
            imgs  = batch['image'].to(device)
            masks = batch['label'].to(device)

            outputs = model(imgs)
            loss    = criterion_bce(outputs, masks) + dice_loss_fn(outputs, masks)
            total_loss += loss.item() * imgs.size(0)

            preds_np = (torch.sigmoid(outputs) > 0.5).cpu().numpy().astype(np.uint8)
            gt_np    = masks.cpu().numpy().astype(np.uint8)

            for i in range(len(preds_np)):
                m = calculate_segmentation_metrics(preds_np[i, 0], gt_np[i, 0])
                for k in all_metrics:
                    all_metrics[k].append(m[k])

    avg_loss    = total_loss / len(loader.dataset)
    avg_metrics = {k: float(np.mean(v)) for k, v in all_metrics.items()}
    return avg_loss, avg_metrics

In [ ]:
seg = EfficientNetUNet(num_classes=1, pretrained=True).to(device)


In [13]:
print("\nLoading best model for test evaluation...")

ckpt = torch.load(
    'models/segmentation/checkpoints/efficientnet_unet_best.pth',
    map_location=device
)

criterion_bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, smooth=1):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1 - dice

seg.load_state_dict(ckpt['model_state_dict'])
seg.eval()

test_loss, test_metrics = evaluate_metrics(
    seg, test_loader, criterion_bce, dice_loss, device
)

print("\n===== TEST RESULTS =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"IoU:       {test_metrics['iou']:.4f}")
print(f"Dice:      {test_metrics['dice']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1:        {test_metrics['f1']:.4f}")


Loading best model for test evaluation...


ValueError: Target size (torch.Size([8])) must be the same as input size (torch.Size([8, 1, 224, 224]))

In [9]:
seg = EfficientNetUNet(num_classes=1, pretrained=True).to(device)

ckpt = torch.load(
    'models/segmentation/checkpoints/efficientnet_unet_best.pth',
    map_location=device
)

seg.load_state_dict(ckpt['model_state_dict'])
seg.eval()
# Tạo dataset test segmentation từ pseudo-label hiện tại nếu có file ground truth không có thì dùng pseudo để đánh nội bộ
# Ở đây dùng same val/test pseudo-file cho demo.
pseudo_dir = 'data/pseudo_labels'
from torchvision import transforms
from torch.utils.data import Dataset
import cv2, numpy as np
class SegTestDataset(Dataset):
    def __init__(self, image_objs, pseudo_dir, transform=None):
        self.image_objs = image_objs
        self.pseudo_dir = pseudo_dir
        self.transform = transform
    def __len__(self):
        return len(self.image_objs)
    def __getitem__(self, idx):
        obj = self.image_objs[idx]  # DurianLeafImage
        img = cv2.imread(obj.image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224,224))
        if self.transform: img = self.transform(img)
        mask_path = os.path.join(pseudo_dir, f'{obj.filename}_pseudo.png')
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (224,224), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((224,224), dtype=np.float32)
        return {'image': img, 'mask': torch.from_numpy(mask).unsqueeze(0), 'filename': obj.filename}

seg_test_ds = SegTestDataset(splits['test'], pseudo_dir, transform=transforms.Compose([transforms.ToPILImage(), transforms.ToTensor()]))
seg_test_loader = TorchDataLoader(seg_test_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())

seg_metrics = {'iou': [], 'dice': [], 'precision': [], 'recall': [], 'f1': []}
with torch.no_grad():
    for batch in seg_test_loader:
        imgs = batch['image'].to(device)
        gt = batch['mask'].to(device)
        preds = torch.sigmoid(seg(imgs)) > 0.5
        preds_np = preds.cpu().numpy().astype(np.uint8)
        gt_np = gt.cpu().numpy().astype(np.uint8)
        for i in range(len(preds_np)):
            m = calculate_segmentation_metrics(preds_np[i,0], gt_np[i,0])
            seg_metrics['iou'].append(m['iou'])
            seg_metrics['dice'].append(m['dice'])
            seg_metrics['precision'].append(m['precision'])
            seg_metrics['recall'].append(m['recall'])
            seg_metrics['f1'].append(m['f1_score'])

seg_final = {k: float(np.mean(v)) for k,v in seg_metrics.items()}
print('Segmentation metrics', seg_final)
os.makedirs('models/segmentation/metrics', exist_ok=True)
with open('models/segmentation/metrics/final_segmentation_metrics.json', 'w') as f:
    json.dump(seg_final, f, indent=2)

RuntimeError: DataLoader worker (pid(s) 12876, 4164, 17396, 17952) exited unexpectedly

## 4) Final report

In [ ]:
report = {
    'classification': clf_metrics,
    'segmentation': seg_final,
    'notes': 'Use pseudo-label as ground truth for segmentation evaluation if true masks unavailable'
}
with open('models/final_evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('Saved final evaluation report to models/final_evaluation_report.json')